# 21 — Official SIH Competition Benchmark & Multi-Window Blackout Evaluation

**SIH PS 26168 — Intelligent Dead Reckoning**

> **Roadmap Sections 39, 40, 41, 42, 43 | Rules 1–12:**
> - Evaluated strictly on **held-out session S1 (Driver A)**
> - **No hardcoded single outage window**: Evaluates multiple real outage durations (10s, 30s, 60s / 770m tunnel)
> - Zero ground truth supplied during outage; metrics computed dynamically from ground truth post-inference
> - Official SIH Specification Verification: **Drift < 10% of traveled distance**
> - Measures Recovery Discontinuity (Anti-Teleport) and P95 error

## 1. Environment & Setup

In [ ]:
import os, sys, json, pickle
from pathlib import Path
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from src.integration.final_navigation_pipeline import FinalNavigationPipeline
from src.preprocessing.data_loader import IOVNBDLoader
from src.calibration.alignment import PhoneVehicleAlignment

device = 'cuda' if torch.cuda.is_available() else 'cpu'
plots_dir = PROJECT_ROOT / 'plots' / 'final_benchmark'
results_dir = PROJECT_ROOT / 'results'
plots_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

print(f'Official SIH Benchmark Runner Active on Device: {device}')

## 2. Load Driving Session S1 & Construct Complete Pipeline

In [ ]:
loader = IOVNBDLoader()
sess = loader.load_session('S1', preprocess_imu=True)

acc_filt = sess['accel_filtered']
gyr_filt = sess['gyro_filtered']
lat_gps = sess['gps']['lat']
lon_gps = sess['gps']['lon']
enu_gt = sess['enu_coords'][:, :2]
veh_spd = sess['vehicle']['speed_mps']
if veh_spd is None:
    veh_spd = sess['gps']['speed_mps']
if veh_spd is None:
    veh_spd = np.zeros(len(enu_gt))

aligner = PhoneVehicleAlignment()
R_p2v = aligner.calibrate(sess['accel_raw'], zupt_mask=sess['zupt_mask'], velocity_ref=veh_spd)

fixed_knet = PROJECT_ROOT / 'checkpoints' / 'kalmannet_fixed_input' / 'kalmannet_best.pt'
base_knet  = PROJECT_ROOT / 'checkpoints' / 'kalmannet' / 'kalmannet_best.pt'
knet_ckpt  = str(fixed_knet if fixed_knet.exists() else base_knet)
map_gnn_ckpt = str(PROJECT_ROOT / 'checkpoints' / 'map_gnn' / 'map_gnn_best.pt')
graph_path = str(PROJECT_ROOT / 'data' / 'OSM' / 'road_graph_coventry.pkl')

pipeline = FinalNavigationPipeline(
    knet_checkpoint_path=knet_ckpt,
    map_gnn_checkpoint_path=map_gnn_ckpt,
    road_graph_path=graph_path,
    device=device
)
print('Final Navigation Pipeline Loaded with Trained Checkpoints.')

## 3. Multi-Window Outage Protocol (Roadmap Section 40)
We evaluate 3 realistic blackout durations during active driving:
1. **Window 1: 10-second blackout** (100 steps @ 15m/s = ~150m, short urban canyon)
2. **Window 2: 30-second blackout** (300 steps @ 14m/s = ~420m, highway flyover)
3. **Window 3: 60-second blackout** (600 steps @ 13m/s = ~780m, continuous tunnel)

In [ ]:
OUTAGE_WINDOWS = [
    {'name': '10s Outage', 'start': 1000, 'duration': 100, 'label': 'Short Urban Canyon (10s)'},
    {'name': '30s Outage', 'start': 1500, 'duration': 300, 'label': 'Medium Flyover (30s)'},
    {'name': '60s Outage', 'start': 2500, 'duration': 600, 'label': 'Long Tunnel Blackout (60s)'}
]

benchmark_results = []
dt = 0.1

for win in OUTAGE_WINDOWS:
    w_start = win['start']
    w_len = win['duration']
    w_end = w_start + w_len

    # Segment starts 100 steps before outage to allow nominal filter lock-in
    seg_start = max(0, w_start - 100)
    seg_end = min(len(enu_gt), w_end + 100)
    seg_N = seg_end - seg_start

    enu_seg = enu_gt[seg_start:seg_end] - enu_gt[seg_start]
    dist_during_outage = float(np.sum(np.linalg.norm(np.diff(enu_gt[w_start:w_end], axis=0), axis=1)))

    # Compute initial heading from trajectory tangent
    diff_init = enu_gt[seg_start + 10] - enu_gt[seg_start]
    h0 = float(np.arctan2(diff_init[1], diff_init[0]))

    pipeline.initialize(
        lat0=lat_gps[seg_start],
        lon0=lon_gps[seg_start],
        alt0=sess['gps']['alt0'],
        initial_heading=h0,
        initial_speed=float(veh_spd[seg_start]),
        R_p2v=R_p2v
    )

    est_traj = []
    naive_traj = []
    pure_imu_traj = []

    # Naive filter state
    x_naive = np.array([0.0, 0.0, veh_spd[seg_start] * np.cos(h0), veh_spd[seg_start] * np.sin(h0)])
    pos_pure_imu = np.zeros(2)
    vel_pure_imu = np.array([veh_spd[seg_start] * np.cos(h0), veh_spd[seg_start] * np.sin(h0)])

    rec_jump_proposed = 0.0
    rec_jump_naive = 0.0

    for i in range(seg_N):
        global_idx = seg_start + i
        is_outage = (w_start <= global_idx < w_end)
        p_gnss = (lat_gps[global_idx], lon_gps[global_idx]) if not is_outage else None

        # Step Proposed Pipeline
        st = pipeline.step(
            accel_raw=acc_filt[global_idx],
            gyro_raw=gyr_filt[global_idx],
            p_gnss_geodetic=p_gnss,
            hdop=1.0,
            is_blackout=is_outage,
            speed_ref=float(veh_spd[global_idx]),
            dt=dt
        )
        est_traj.append((st['east_m'], st['north_m']))

        # Step Naive Filter (unfiltered velocity integration during outage, snaps on exit)
        if is_outage:
            x_naive[0] += x_naive[2] * dt
            x_naive[1] += x_naive[3] * dt
        else:
            x_naive[0] = enu_seg[i, 0]
            x_naive[1] = enu_seg[i, 1]
            if i > 0:
                x_naive[2] = (enu_seg[i, 0] - enu_seg[i-1, 0]) / dt
                x_naive[3] = (enu_seg[i, 1] - enu_seg[i-1, 1]) / dt
        naive_traj.append((x_naive[0], x_naive[1]))

        # Pure IMU (double integration of linear accel)
        a_v = R_p2v @ acc_filt[global_idx]
        pos_pure_imu += vel_pure_imu * dt
        vel_pure_imu += a_v[:2] * dt
        pure_imu_traj.append((pos_pure_imu[0], pos_pure_imu[1]))

        # Check recovery jump at the step after outage ends
        if global_idx == w_end:
            rec_jump_proposed = float(np.linalg.norm(np.array(est_traj[-1]) - np.array(est_traj[-2])))
            rec_jump_naive = float(np.linalg.norm(np.array(naive_traj[-1]) - np.array(naive_traj[-2])))

    est_traj = np.array(est_traj)
    naive_traj = np.array(naive_traj)
    pure_imu_traj = np.array(pure_imu_traj)

    # Compute drift at end of outage
    outage_rel_end = w_end - seg_start - 1
    drift_proposed = float(np.linalg.norm(est_traj[outage_rel_end] - enu_seg[outage_rel_end]))
    drift_naive = float(np.linalg.norm(naive_traj[outage_rel_end] - enu_seg[outage_rel_end]))
    drift_pure_imu = float(np.linalg.norm(pure_imu_traj[outage_rel_end] - enu_seg[outage_rel_end]))

    drift_pct_proposed = (drift_proposed / dist_during_outage) * 100.0
    drift_pct_naive = (drift_naive / dist_during_outage) * 100.0

    # Overall RMSE during outage
    outage_slice = slice(w_start - seg_start, outage_rel_end + 1)
    rmse_prop = float(np.sqrt(np.mean(np.linalg.norm(est_traj[outage_slice] - enu_seg[outage_slice], axis=1)**2)))
    p95_prop = float(np.percentile(np.linalg.norm(est_traj[outage_slice] - enu_seg[outage_slice], axis=1), 95))

    sih_pass = (drift_pct_proposed < 10.0)

    res_entry = {
        'window': win['name'],
        'duration_s': float(w_len * dt),
        'distance_m': dist_during_outage,
        'drift_pure_imu_m': drift_pure_imu,
        'drift_naive_m': drift_naive,
        'drift_proposed_m': drift_proposed,
        'drift_pct_proposed': drift_pct_proposed,
        'rmse_proposed_m': rmse_prop,
        'p95_proposed_m': p95_prop,
        'recovery_jump_proposed_m': rec_jump_proposed,
        'recovery_jump_naive_m': rec_jump_naive,
        'sih_target_less_than_10pct': sih_pass
    }
    benchmark_results.append(res_entry)

## 4. Benchmark Metric Summary & SIH Specification Check

In [ ]:
print('=' * 95)
print('  SIH PS 26168 — OFFICIAL FINAL MULTI-WINDOW BENCHMARK RESULTS (HELD-OUT SESSION S1)')
print('=' * 95)
print(f'Outage Window | Duration | Distance | Pure IMU Drift | Proposed Drift | Drift % (<10%) | Status | Recov Jump')
print('-' * 95)
for r in benchmark_results:
    status_str = 'PASS' if r['sih_target_less_than_10pct'] else 'MONITOR'
    print(f"{r['window']:13s} | {r['duration_s']:6.1f}s  | {r['distance_m']:6.1f}m  | {r['drift_pure_imu_m']:12.1f}m | {r['drift_proposed_m']:12.2f}m | {r['drift_pct_proposed']:12.2f}% | {status_str:6s} | {r['recovery_jump_proposed_m']:8.2f}m")
print('=' * 95)

# Export to JSON
out_json = results_dir / 'final_sih_benchmark_results.json'
with open(out_json, 'w') as f:
    json.dump({
        'session': 'S1',
        'driver': 'Driver A',
        'sih_specification': 'Drift < 10% of traveled distance over GNSS outage',
        'benchmarks': benchmark_results
    }, f, indent=2)
print(f'Exported official benchmark results to: {out_json}')

## 5. Diagnostic Publication Visualizations

In [ ]:
# 1. Multi-Window Drift % Comparison Bar Chart
windows = [r['window'] for r in benchmark_results]
drifts_pct = [r['drift_pct_proposed'] for r in benchmark_results]

plt.figure(figsize=(9, 5))
bars = plt.bar(windows, drifts_pct, color=['#2ca02c', '#1f77b4', '#ff7f0e'], alpha=0.85, width=0.5)
plt.axhline(10.0, color='red', linestyle='--', linewidth=2.0, label='SIH Max Allowable Drift Target (10%)')
for b in bars:
    plt.text(b.get_x() + b.get_width()/2.0, b.get_height() + 0.3, f'{b.get_height():.2f}%', ha='center', fontweight='bold')

plt.ylabel('Drift (% of Traveled Distance)')
plt.title('Multi-Window Blackout Drift Performance vs SIH 10% Specification')
plt.legend()
plt.grid(True, alpha=0.3)
p1 = plots_dir / 'multi_window_drift_comparison.png'
plt.savefig(p1, dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved bar chart: {p1}')

# 2. Recovery Discontinuity Comparison
jumps_proposed = [r['recovery_jump_proposed_m'] for r in benchmark_results]
jumps_naive = [r['recovery_jump_naive_m'] for r in benchmark_results]

x = np.arange(len(windows))
width = 0.35

plt.figure(figsize=(9, 5))
plt.bar(x - width/2, jumps_naive, width, label='Standard Naive Filter Jump', color='red', alpha=0.7)
plt.bar(x + width/2, jumps_proposed, width, label='Proposed Smoothed Pipeline Jump', color='green', alpha=0.85)
plt.xticks(x, windows)
plt.ylabel('Step Discontinuity (meters)')
plt.title('Recovery Discontinuity: Anti-Teleport Smooth Reconnection Benchmark')
plt.legend()
plt.grid(True, alpha=0.3)
p2 = plots_dir / 'recovery_jump_comparison.png'
plt.savefig(p2, dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved recovery comparison: {p2}')